In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from scripts.preprocessing_utils import run_full_preprocessing_pipeline

df_raw = pd.read_csv("../data/Divar.csv")
df_clean = run_full_preprocessing_pipeline(df_raw)

if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

df_clean.to_parquet("../data/Divar_cleaned.parquet", index=False)
print("shape of data set :", df_clean.shape)

/tmp/ipykernel_47109/226645954.py:9: DtypeWarning: Columns (11: rent_to_single, 27: floor, 29: total_floors_count, 53: extra_person_capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../data/Divar.csv")


shape of data set : (999997, 62)


In [2]:
import pandas as pd

new_df = pd.read_parquet("../data/Divar_cleaned.parquet")

In [3]:
print(new_df.columns.tolist())


['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug', 'user_type', 'description', 'title', 'rent_mode', 'rent_value', 'rent_type', 'price_mode', 'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform', 'transformable_price', 'transformable_credit', 'transformed_credit', 'transformable_rent', 'transformed_rent', 'building_size', 'deed_type', 'floor', 'rooms_count', 'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt', 'has_warm_water_provider', 'has_heating_system', 'has_cooling_system', 'has_restroom', 'building_direction', 'floor_material', 'regular_person_capacity', 'extra_person_capacity', 'cost_per_extra_person', 'rent_price_on_regular_days', 'rent_price_on_special_days', 'rent_price_at_weekends', 'location_latitude', 'location_longitude', 'has_land', 'cost_per_extra_person_suspicious_low', 'rent_price_on_regular_days_suspicious_low', 'rent_price_on_special_days_suspicious_low',

In [4]:
import pandas as pd
import numpy as np

# ۱. بارگذاری دیتاست
df = pd.read_parquet("../data/Divar_cleaned.parquet")

# ۲. ساخت جدول بازرسی کیفیت برای تمام ۶۱ ستون
audit_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique(),
    'Sample Value': [df[col].dropna().iloc[0] if not df[col].dropna().empty else None for col in df.columns]
})

# تنظیم نمایش پایتون برای دیدن تمام ۶۱ سطر
pd.set_option('display.max_rows', 100)
print(f"📊 چک‌لیست سلامت کل ویژگی‌ها (مجموعاً {df.shape[1]} ستون):")
display(audit_df)

📊 چک‌لیست سلامت کل ویژگی‌ها (مجموعاً 62 ستون):


,Data Type,Null Count,Null %,Unique Values,Sample Value
cat2_slug,string,0,0.00,6,temporary-rent
cat3_slug,string,0,0.00,16,villa
city_slug,string,0,0.00,421,karaj
neighborhood_slug,string,0,0.00,1189,mehrshahr
user_type,string,0,0.00,3,مشاور املاک
description,string,1,0.00,988816,۵۰۰متر ۲۰۰متر بنا دوبلکس ۳خواب استخر آبگرم داخ...
title,string,54,0.01,902755,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه
rent_mode,string,647005,64.70,3,مقطوع
rent_value,float64,648676,64.87,1311,26000000.0
rent_type,string,896036,89.60,2,rent_credit


## تغییرات ستون‌ها در مرحله پیش‌پردازش (Preprocessing Summary)

---

### ۱. ستون‌های حذف‌شده (Removed Columns)

| نام ستون | دلیل حذف / توضیحات |
| :--- | :--- |
| `property_type` | تداخل با دسته‌بندی‌های اصلی (`cat2_slug` و `cat3_slug`) |
| `land_size` | درصد بسیار بالای مقادیر مفقوده (Null) |
| `has_business_deed` | داده مفقوده بالاتر از حد مجاز (بیش از ۹۰٪) |
| `has_water` | عدم تمایزپذیری مناسب و درصد بالای مقادیر مفقوده |
| `has_electricity` | عدم تمایزپذیری مناسب و درصد بالای مقادیر مفقوده |
| `has_gas` | عدم تمایزپذیری مناسب و درصد بالای مقادیر مفقوده |
| `has_security_guard` | عدم ثبت در اکثر آگهی‌ها (عدم پوشش کافی) |
| `has_barbecue` | عدم ثبت در اکثر آگهی‌ها (عدم پوشش کافی) |
| `has_pool` | عدم ثبت در اکثر آگهی‌ها (عدم پوشش کافی) |
| `has_jacuzzi` / `has_sauna` | عدم ثبت در اکثر آگهی‌ها (عدم پوشش کافی) |
| `location_radius` | جایگزینی با مختصات استاندارد دکارتی (UTM) |
| `created_at_month` | بازسازی و جایگزینی با فرمت استاندارد `created_month` |
| `rent_to_single` | نال بالای ۹۹٪ (تقریباً بدون پوشش داده) |

---

### ۲. ستون‌های جدید ساخته‌شده (Added Features)

| نام ستون جدید | نوع داده | توضیحات و نحوه ساخت |
| :--- | :--- | :--- |
| `has_land` | Boolean | نشان‌دهنده وجود متراژ زمین یا ویلایی بودن ملک |
| `title_length` | Integer | تعداد کاراکترهای عنوان آگهی |
| `title_word_count` | Integer | تعداد کلمات عنوان آگهی |
| `description_length` | Integer | تعداد کاراکترهای متن توضیحات آگهی |
| `description_word_count` | Integer | تعداد کلمات متن توضیحات آگهی |
| `cost_per_extra_person_suspicious_low` | Boolean Flag | شناساگر قیمت‌های اجاره روزانه مشکوک به خطا |
| `rent_price_on_regular_days_suspicious_low` | Boolean Flag | شناساگر قیمت روزهای عادی مشکوک به خطا |
| `rent_price_on_special_days_suspicious_low` | Boolean Flag | شناساگر قیمت روزهای خاص مشکوک به خطا |
| `rent_price_at_weekends_suspicious_low` | Boolean Flag | شناساگر قیمت آخر هفته مشکوک به خطا |
| `transformed_credit_suspicious_low` | Boolean Flag | شناساگر مقادیر مشکوک به خطا در `transformed_credit` (مقادیر خیلی پایین) |
| `transformed_rent_suspicious_low` | Boolean Flag | شناساگر مقادیر مشکوک به خطا در `transformed_rent` (مقادیر خیلی پایین) |
| `inferred_full_credit` | Boolean | محاسبه تخمینی رهن کامل بر اساس فرمول تبدیل رهن و اجاره |
| `location_utm_x` | Float | تبدیل `location_latitude` به مختصات متری UTM (X) |
| `location_utm_y` | Float | تبدیل `location_longitude` به مختصات متری UTM (Y) |
| `created_year` | Integer | سال ثبت آگهی |
| `created_month` | Integer | ماه ثبت آگهی |